# 03 · Visualize

讀 `data/processed/village_to_nearest_library.csv` + 里界 GeoJSON，輸出：
- `output/maps/tainan_library_time_static.png`
- `output/maps/tainan_library_time_interactive.html`

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from lib.colors import BINS_MINUTES, COLORS_HEX

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "output" / "maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

villages = gpd.read_file(RAW_DIR / "tainan_villages.geojson")
times = pd.read_csv(PROC_DIR / "village_to_nearest_library.csv", dtype={"village_id": str})
libs = pd.read_csv(RAW_DIR / "tainan_libraries.csv")

merged = villages.merge(times[["village_id", "drive_minutes", "nearest_library", "method"]], on="village_id", how="left")
print(f"Merged: {len(merged)} villages, missing time: {merged['drive_minutes'].isna().sum()}")
merged.head()


In [ ]:
cmap = ListedColormap(COLORS_HEX)
bounds = [0, *BINS_MINUTES, 1e9]
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(12, 14), dpi=150)

merged.plot(
    column="drive_minutes",
    cmap=cmap,
    norm=norm,
    edgecolor="white",
    linewidth=0.15,
    ax=ax,
    missing_kwds={"color": "lightgray", "label": "no data"},
)

# 區界（粗一點）
districts = merged.dissolve(by="district", as_index=False)
districts.boundary.plot(ax=ax, color="black", linewidth=0.6)

# 圖書館位置
libs_gdf = gpd.GeoDataFrame(libs, geometry=gpd.points_from_xy(libs.lon, libs.lat), crs="EPSG:4326")
libs_gdf.plot(ax=ax, marker="P", color="black", markersize=40, edgecolor="white", linewidth=0.5)

# 圖例
labels = ["0–5", "5–10", "10–15", "15–20", "20–30", "30+"]
legend_handles = [Patch(facecolor=c, edgecolor="white", label=f"{lab} 分鐘") for c, lab in zip(COLORS_HEX, labels)]
legend_handles.append(Patch(facecolor="lightgray", edgecolor="white", label="無資料"))
ax.legend(handles=legend_handles, title="到最近市立圖書館", loc="lower left", fontsize=9)

# 中文標題：matplotlib 預設字型可能無中文 → 用系統字型；找不到可註解掉
try:
    plt.rcParams["font.sans-serif"] = ["PingFang TC", "Heiti TC", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

ax.set_title("台南市各里到最近市立圖書館行車時間", fontsize=14, pad=10)
ax.set_axis_off()
ax.set_aspect("equal")

out = OUTPUT_DIR / "tainan_library_time_static.png"
fig.savefig(out, bbox_inches="tight", dpi=150)
plt.show()
print(f"✅ Saved {out}")


In [ ]:
import folium
from folium.features import GeoJsonTooltip
from lib.colors import minutes_to_color

# 為每個 feature 加 fill_color 屬性
merged["fill_color"] = merged["drive_minutes"].apply(
    lambda m: minutes_to_color(m) if pd.notna(m) else "#cccccc"
)
merged["drive_minutes_str"] = merged["drive_minutes"].apply(
    lambda m: f"{m:.1f}" if pd.notna(m) else "N/A"
)

# 地圖中心：台南幾何中心
centroid = merged.geometry.unary_union.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(
    merged.to_json(),
    name="到最近圖書館行車時間",
    style_function=lambda feat: {
        "fillColor": feat["properties"]["fill_color"],
        "color": "white",
        "weight": 0.3,
        "fillOpacity": 0.75,
    },
    tooltip=GeoJsonTooltip(
        fields=["village_name", "district", "nearest_library", "drive_minutes_str", "method"],
        aliases=["里", "區", "最近圖書館", "預估時間(分)", "計算方式"],
        sticky=True,
    ),
).add_to(m)

# 圖書館標記
for _, lib in libs.iterrows():
    folium.Marker(
        location=[lib["lat"], lib["lon"]],
        popup=folium.Popup(f'<b>{lib["name"]}</b><br>{lib["district"]}<br>{lib["address"]}', max_width=300),
        icon=folium.Icon(color="black", icon="book", prefix="fa"),
    ).add_to(m)

# 圖例（HTML 直接嵌入）
legend_html = (
    '<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;'
    ' background: white; padding: 10px; border: 1px solid #999;'
    ' font-family: sans-serif; font-size: 12px;">'
    '  <b>到最近市立圖書館（分鐘）</b><br>'
'<div><span style="display:inline-block;width:14px;height:14px;background:#1a9641;margin-right:6px;"></span>0–5</div><div><span style="display:inline-block;width:14px;height:14px;background:#a6d96a;margin-right:6px;"></span>5–10</div><div><span style="display:inline-block;width:14px;height:14px;background:#ffffbf;margin-right:6px;"></span>10–15</div><div><span style="display:inline-block;width:14px;height:14px;background:#fdae61;margin-right:6px;"></span>15–20</div><div><span style="display:inline-block;width:14px;height:14px;background:#d7191c;margin-right:6px;"></span>20–30</div><div><span style="display:inline-block;width:14px;height:14px;background:#7f0000;margin-right:6px;"></span>30+</div>'
    + '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))

out = OUTPUT_DIR / "tainan_library_time_interactive.html"
m.save(str(out))
print(f"✅ Saved {out}")
m
